# Data Description
- Author: RUEI-HSIANG, CHANG（B12303030；AI協助）
- Date first created: 20261019
- Background: 10人政府具名人物／職務名錄新增入口，其中9人新增完整生日、性別、2022學歷級別及非首次任何公職；黃宣寓只明載男與簡歷。尚非10份完整背景。
- Unit of observation: master/council/review為候選人；fields為候選人×欄位；evidence為候選人×欄位×來源。
- Sample period: 2026候選人名冊框架，2022跨公職申報與具日期限制的政府人物／名錄頁。
- Known issues: 入口缺720人不代表其餘已補齊學經歷。707→716人有政府完整生日支持，717人性別有明載來源聯集；703人2022議員申報序列未變。其他公職不能判作曾參選議員，9人議員首次仍未知。縣市長生日22人與魏造文性別未齊；社群、Meta與窗口本輪未變。沒有找到不填首次=1。
- Definition for each variable: 沿用[前輪完整schema與資料字典](cross_office_round8_20261019_documentation.ipynb)。本輪additional_profile_status明記government_person_or_role_entry，區分名錄與完整人物履歷；birth_date_additional_official及來源／年份為2022其他公職申報；sex_additional_source_value為明載性別，黃宣寓來源年未明列；education_raw與additional_historical_education_cec僅2022申報級別；experience_raw為具名段落審閱摘要，非完整生涯。review的context/person_anchor、cec_file/row/locality與decision保留逐人身分依據；evidence可一人多欄多來源。
- Data path / Downstream analysis input: 下方5個精確檔，角色review_inventory/evidence，尚非最終研究樣本。
- Source data: SOURCE_PINS完整固定清單與hash，包括3個pre_municipal_round9快照、人工核對設定、原始CEC與政府名錄、來源metadata；不把上游工作單主張當真值。
- Processing code: [municipal_roles_round9_20261019.ipynb](../code/municipal_roles_round9_20261019.ipynb)。
- Role: review inventory / evidence。
- Verification status: 下方保存檔讀回、schema、鍵、SHA與未變欄位檢查；不執行下載或寫入。
- SHA-256: 逐檔如下。
- `data/processed/candidate_master_current.csv`：1583列／34欄；SHA-256 `960d538e9af2c15430f4c25f20c523f0e752f2fa16b5492f00b6f9d8c4e65dd4`。
- `data/processed/candidate_fields_current.csv`：31660列／11欄；SHA-256 `3d0a35feebd892772286c646ae7b3b0c3bf8e1a0d7f2d0b74c7825f253b9fe46`。
- `data/processed/council_background_20261019.csv`：1502列／50欄；SHA-256 `1f2af881749b1c5e06620d85a3a36261c08c9e84d64805eb6159e11ff9013fdd`。
- `data/processed/municipal_roles_review_round9_20261019.csv`：10列／18欄；SHA-256 `142bf3e8e4a55c7369274cf35c9526edb2fc86c4daf68adb0b231a3c98d49234`。
- `data/processed/candidate_evidence_round9_20261019.csv`：61列／9欄；SHA-256 `3ea1d9b42882bb092b2983904d35c9deebb1c967ba51102b2be388e9fe5b01d7`。

In [1]:
from pathlib import Path
import pandas as pd,json,hashlib,csv
from bs4 import BeautifulSoup
from IPython.display import display
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'AGENTS.md').exists() and (p/'data').exists())
sha=lambda p:hashlib.sha256((ROOT/p).read_bytes()).hexdigest()
SOURCE_PINS={'data/processed/snapshots/pre_municipal_round9_20261019/candidate_master_current.csv': '4fba9288b89aa417734ea686b09ccad5eed147b47162265c799319bd025fa926', 'data/processed/snapshots/pre_municipal_round9_20261019/candidate_fields_current.csv': '80d9ccb7ea2cb1e5adfd3488f0b3d16581b067311d9e230f3daee37cf7b059ad', 'data/processed/snapshots/pre_municipal_round9_20261019/council_background_20261019.csv': 'd6542846ba72ae53b3ce5780f62daa99199ccd28faadaf3bf15bfe6b19106737', 'data/code/municipal_roles_review_round9_20261019.json': 'ce45941f035646008ec98a41aaaa337d8c1819752c2d853000e5ef67bf769e56', 'data/processed/council_career_source_worklist_round6_20261019.csv': '171f8bfcfa268720d1d36c9928092ecc80e04457a6709172823e31c213717620', 'data/raw/cec_all_offices_manifest.json': '726d464020ff8f850ce39685bb70926c8529712f75c1833ac5b5c6d795354ed7', 'data/raw/cec_all_offices/101_elcand.csv': '35853ac9d8a7eee4e9882bc04148462876110dcdb3cf0cdc11af0908e1c800f0', 'data/raw/cec_all_offices/103_elcand.csv': '46773806f1b6ef6389e632be7c10ea59bcb5c3afff3ad2715787974eafe494f0', 'data/raw/cec_all_offices/112_elcand.csv': 'f355705254d5ca457d4fe130b77983b769602126b10713e7c968889d2d52530c', 'data/raw/cross_office_round8_20261019/2022_village_elbase.csv': '4d0afbae084ccd77785ebe9a878eadb7e97e828555926d92866dddbfb4b14e79', 'data/raw/municipal_roles_round9_20261019/82e3fb55c475911e453d.html': 'ee5eccb74a99bc10078257e62d2a98fc4986eab743c853b7ce5f9debb56ab3e3', 'data/raw/municipal_roles_round9_20261019/6556882ec6ac55681d90.html': '027ddae3c619ea334c71cf921f67e0a119822e547f351f40b894d9a37eb12a9e', 'data/raw/municipal_roles_round9_20261019/d56e6a8a8d5ccaf32087.html': 'e8aa0d5deac638ff4b1be04646df7c2795520e1b2d2c38fe44bca6b5e460be09', 'data/raw/municipal_roles_round9_20261019/2022_R1_elbase.csv': '96ee98216f80e97cee27c8edbbbab35c240717bdd2ecf5eef4edaea4b45d36ee', 'data/raw/municipal_roles_round9_20261019/2022_D1_elbase.csv': '780f22895ffc9db630f92733fce3defbe913e6c936ca64cecf8860e34df0768a', 'data/raw/municipal_roles_round9_20261019/manifest.json': 'b32a0ca59f85bfd184e2f81af88e5c416430cc927196b280f238d31b7c2fb820', 'data/raw/municipal_roles_round9_20261019/extraction_manifest.json': '7f036b1e2f81e98e11cd54e36505f2c67b12ffc29ac7683dba8885a58c113b5f', 'data/raw/municipal_roles_round9_20261019/tpp_zhuang_web_excerpt.json': '02bb213dd07af91d2090223b74aea4164f51d02196faab68f5a3e0fe976c2a4f', 'data/raw/career_expansion_20261019/b587a5e46c6e1f6751ab.source': '546bf7a06c22f241135b6d64454d4d79615684ec3cb5f7679f053b59361311de', 'data/raw/career_expansion_20261019/e3a22c5d3489daf8cda3.source': '6ef3258faceba1559d55f5b3b33bedb3b5c5913ff69785351ffb4e7130fd67fb', 'data/raw/career_expansion_20261019/ae8adaeaad8ae7afaa36.source': '4cf7c9437b56e9757ba167f8f37a7e8b9375beac6b35b0e4566509fe28c322da', 'data/raw/career_expansion_20261019/64937ab1e916ff2df53f.source': 'a82ac876aea775810ba88270c917de751a00500e0b5af08d504ff6fbcc85b67e', 'data/raw/career_expansion_20261019/90edbc3b20226a41f3a8.source': 'ccc5959c2ab818c43776497d2c8e631553a2203097dabd8b1932f06063a6b66e', 'data/raw/career_expansion_20261019/0912e18a119d299531bf.source': 'b9f023b3b2aa30d147b291e3eaaa717987ac5ad5a381bb3ddde304ebaba73784', 'data/raw/career_expansion_20261019/606002eb4a668a9a18ea.source': '6a0a025642a80743ce91058c250cb8e3ae1e724569209f26667bd409624c8108', 'data/raw/career_expansion_20261019/6b8915ef8fb902feabfa.source': '6b20ad535e95fba23c662a8db3c4de71ef6d5677e95b9f2b93bcb02df11f67b9', 'data/raw/career_expansion_20261019/8fb25c97b1da43ae8bd0.source': 'dbd04ba156909aa18bc3e3d8e597a16d8a02a6ec2b13adb66ad1d0695633c019'}
EXPECTED={'data/processed/candidate_master_current.csv': {'rows': 1583, 'columns': 34, 'key': ['candidate_key'], 'sha256': '960d538e9af2c15430f4c25f20c523f0e752f2fa16b5492f00b6f9d8c4e65dd4'}, 'data/processed/candidate_fields_current.csv': {'rows': 31660, 'columns': 11, 'key': ['candidate_key', 'field'], 'sha256': '3d0a35feebd892772286c646ae7b3b0c3bf8e1a0d7f2d0b74c7825f253b9fe46'}, 'data/processed/council_background_20261019.csv': {'rows': 1502, 'columns': 50, 'key': ['candidate_key'], 'sha256': '1f2af881749b1c5e06620d85a3a36261c08c9e84d64805eb6159e11ff9013fdd'}, 'data/processed/municipal_roles_review_round9_20261019.csv': {'rows': 10, 'columns': 18, 'key': ['candidate_key'], 'sha256': '142bf3e8e4a55c7369274cf35c9526edb2fc86c4daf68adb0b231a3c98d49234'}, 'data/processed/candidate_evidence_round9_20261019.csv': {'rows': 61, 'columns': 9, 'key': [], 'sha256': '3ea1d9b42882bb092b2983904d35c9deebb1c967ba51102b2be388e9fe5b01d7'}}
for p,h in SOURCE_PINS.items():assert sha(p)==h,p
D={}
for p,s in EXPECTED.items():
 assert sha(p)==s['sha256'],p
 d=pd.read_csv(ROOT/p,dtype=str,keep_default_na=False);assert d.shape==(s['rows'],s['columns'])
 if s['key']:assert not d.duplicated(s['key']).any()
 D[p]=d
print('Exact inputs and saved output versions checked:',len(SOURCE_PINS),len(D))

Exact inputs and saved output versions checked: 27 5


## Change for Your Own Data
讀取固定保存檔，不重新處理。

## Summary

### Sample

In [2]:
for p,d in D.items():print(p);display(d.head(10))

data/processed/candidate_master_current.csv


,candidate_key,candidate_name,office_type,county_city,electoral_district,party_registration_raw,registration_source,registration_source_file,registration_source_page,qualification_inherited,...,Threads_status,posts_Facebook_status,posts_Instagram_status,posts_Threads_status,followers_status,policy_text_status,visual_status,political_family_status,administrative_experience_status,qualification_status
0,2026-C1-001,郭璽,直轄市長,臺北市,,台灣麻將最大黨,https://web.cec.gov.tw/api/file/bb9a8d7a-9b8a-...,data/raw/2026_metropolitan_registration.pdf,1,approved_2026-10-16,...,尚未驗證，不代表無帳號,未取得，不能視為零,未取得，不能視為零,未取得，不能視為零,未取得，不能視為零,待補查原文,未查找,未查找,未查找,官方內文待重新核對
1,2026-C1-002,沈伯洋,直轄市長,臺北市,,民主進步黨,https://web.cec.gov.tw/api/file/bb9a8d7a-9b8a-...,data/raw/2026_metropolitan_registration.pdf,1,approved_2026-10-16,...,人物頁／競選頁已列帳號連結；目標內容待核對,未取得，不能視為零,未取得，不能視為零,未取得，不能視為零,未取得，不能視為零,部分原文已保存，尚未政策編碼,未查找,未查找,未查找,官方內文待重新核對
2,2026-C1-003,蕭文乾,直轄市長,臺北市,,臺灣SoR無法黨,https://web.cec.gov.tw/api/file/bb9a8d7a-9b8a-...,data/raw/2026_metropolitan_registration.pdf,1,approved_2026-10-16,...,尚未驗證，不代表無帳號,未取得，不能視為零,未取得，不能視為零,未取得，不能視為零,未取得，不能視為零,待補查原文,未查找,未查找,未查找,官方內文待重新核對
3,2026-C1-004,唐新民,直轄市長,臺北市,,三勢團結促進聯盟,https://web.cec.gov.tw/api/file/bb9a8d7a-9b8a-...,data/raw/2026_metropolitan_registration.pdf,1,approved_2026-10-16,...,尚未驗證，不代表無帳號,未取得，不能視為零,未取得，不能視為零,未取得，不能視為零,未取得，不能視為零,待補查原文,未查找,未查找,未查找,官方內文待重新核對
4,2026-C1-005,蔣萬安,直轄市長,臺北市,,中國國民黨,https://web.cec.gov.tw/api/file/bb9a8d7a-9b8a-...,data/raw/2026_metropolitan_registration.pdf,1,approved_2026-10-16,...,尚未驗證，不代表無帳號,未取得，不能視為零,未取得，不能視為零,未取得，不能視為零,未取得，不能視為零,待補查原文,未查找,未查找,未查找,官方內文待重新核對
5,2026-C1-006,林志成,直轄市長,臺北市,,無,https://web.cec.gov.tw/api/file/bb9a8d7a-9b8a-...,data/raw/2026_metropolitan_registration.pdf,1,approved_2026-10-16,...,尚未驗證，不代表無帳號,未取得，不能視為零,未取得，不能視為零,未取得，不能視為零,未取得，不能視為零,待補查原文,未查找,未查找,未查找,官方內文待重新核對
6,2026-C1-007,蘇巧慧,直轄市長,新北市,,民主進步黨,https://web.cec.gov.tw/api/file/bb9a8d7a-9b8a-...,data/raw/2026_metropolitan_registration.pdf,1,approved_2026-10-16,...,人物頁／競選頁已列帳號連結；目標內容待核對,未取得，不能視為零,未取得，不能視為零,未取得，不能視為零,未取得，不能視為零,部分原文已保存，尚未政策編碼,未查找,未查找,未查找,官方內文待重新核對
7,2026-C1-008,李四川,直轄市長,新北市,,中國國民黨,https://web.cec.gov.tw/api/file/bb9a8d7a-9b8a-...,data/raw/2026_metropolitan_registration.pdf,1,approved_2026-10-16,...,人物頁／競選頁已列帳號連結；目標內容待核對,未取得，不能視為零,未取得，不能視為零,未取得，不能視為零,未取得，不能視為零,待補查原文,未查找,未查找,未查找,官方內文待重新核對
8,2026-C1-009,蘇輝湟,直轄市長,新北市,,無,https://web.cec.gov.tw/api/file/bb9a8d7a-9b8a-...,data/raw/2026_metropolitan_registration.pdf,1,approved_2026-10-16,...,尚未驗證，不代表無帳號,未取得，不能視為零,未取得，不能視為零,未取得，不能視為零,未取得，不能視為零,待補查原文,未查找,未查找,未查找,官方內文待重新核對
9,2026-C1-010,張善政,直轄市長,桃園市,,中國國民黨,https://web.cec.gov.tw/api/file/bb9a8d7a-9b8a-...,data/raw/2026_metropolitan_registration.pdf,1,approved_2026-10-16,...,尚未驗證，不代表無帳號,未取得，不能視為零,未取得，不能視為零,未取得，不能視為零,未取得，不能視為零,待補查原文,未查找,未查找,未查找,官方內文待重新核對


data/processed/candidate_fields_current.csv


,candidate_key,candidate_name,office_type,county_city,field,status,existing_value,evidence,note,next_source,updated_date
0,2026-C1-001,郭璽,直轄市長,臺北市,sex,已有資料，待來源複核,男,https://db.cec.gov.tw/,dob_corroborated_by_media,議會／政府人物頁、歷屆選舉公報；身分與時間核對,2026-10-19
1,2026-C1-001,郭璽,直轄市長,臺北市,birth_date,已有資料，待來源複核,1958-08-31,https://db.cec.gov.tw/,CEC_name_and_birth_date_corroborated_by_CNA,議會／政府人物頁、歷屆選舉公報；身分與時間核對,2026-10-19
2,2026-C1-001,郭璽,直轄市長,臺北市,education,已有資料，待來源複核,大學,,歷史公報或媒體補充，非2026學歷認證,議會／政府人物頁、歷屆選舉公報；身分與時間核對,2026-10-19
3,2026-C1-001,郭璽,直轄市長,臺北市,incumbency,既有搜尋後仍缺，待補查,,,,議會／政府人物頁、歷屆選舉公報；身分與時間核對,2026-10-19
4,2026-C1-001,郭璽,直轄市長,臺北市,first_time_this_office,既有搜尋後仍缺，待補查,,,0=有過參選證據；空白不表示首次。此處沿用既有表。,議會／政府人物頁、歷屆選舉公報；身分與時間核對,2026-10-19
5,2026-C1-001,郭璽,直轄市長,臺北市,first_time_any_office,已有資料，待來源複核,0,https://db.cec.gov.tw/,0=有過參選證據；空白不表示首次。此處沿用既有表。,議會／政府人物頁、歷屆選舉公報；身分與時間核對,2026-10-19
6,2026-C1-001,郭璽,直轄市長,臺北市,profile,既有搜尋後仍缺，待補查,,,,議會／政府人物頁、歷屆選舉公報；身分與時間核對,2026-10-19
7,2026-C1-001,郭璽,直轄市長,臺北市,website,既有搜尋後仍缺，待補查,,,,議會／政府人物頁、歷屆選舉公報；身分與時間核對,2026-10-19
8,2026-C1-001,郭璽,直轄市長,臺北市,Facebook,尚未驗證，不代表無帳號,,,,官方人物頁／競選網站交叉連結,2026-10-19
9,2026-C1-001,郭璽,直轄市長,臺北市,Instagram,尚未驗證，不代表無帳號,,,,官方人物頁／競選網站交叉連結,2026-10-19


data/processed/council_background_20261019.csv


,candidate_key,candidate_name,county_city,office_type,electoral_district,party_registration_raw,birth_date_supported_history,birth_year_supported_history,sex_supported_history,birth_sex_evidence_status,...,bulletin_source_page,bulletin_identity_status,birth_date_additional_official,birth_date_additional_source_url,birth_date_additional_evidence_status,round6_profile_urls,birth_date_additional_source_year,additional_historical_education_cec,round7_identity_url,party_page_lead_url
0,2026|直轄市議員|臺北市第1選舉區|謝閔弘,謝閔弘,臺北市,直轄市議員,臺北市第1選舉區,時代力量,,,,尚無可連結的歷史紀錄,...,,,,,,,,,,
1,2026|直轄市議員|臺北市第1選舉區|李文,李文,臺北市,直轄市議員,臺北市第1選舉區,無,,,,尚無可連結的歷史紀錄,...,,,,,,,,,,
2,2026|直轄市議員|臺北市第1選舉區|陳彥廷,陳彥廷,臺北市,直轄市議員,臺北市第1選舉區,無,,,,尚無可連結的歷史紀錄,...,,,,,,,,,,
3,2026|直轄市議員|臺北市第1選舉區|鍾佩玲,鍾佩玲,臺北市,直轄市議員,臺北市第1選舉區,民主進步黨,1979-03-27,1979,女,2022官方申報，先前身分連結審核有支持；非2026戶籍認定,...,,,,,,,,,,
4,2026|直轄市議員|臺北市第1選舉區|紀建漢,紀建漢,臺北市,直轄市議員,臺北市第1選舉區,無,,,,尚無可連結的歷史紀錄,...,,,,,,,,,,
5,2026|直轄市議員|臺北市第1選舉區|陳賢蔚,陳賢蔚,臺北市,直轄市議員,臺北市第1選舉區,民主進步黨,1977-07-12,1977,男,2022官方申報，先前身分連結審核有支持；非2026戶籍認定,...,,,,,,,,,,
6,2026|直轄市議員|臺北市第1選舉區|林世宗,林世宗,臺北市,直轄市議員,臺北市第1選舉區,民主進步黨,1959-06-13,1959,男,2022官方申報，先前身分連結審核有支持；非2026戶籍認定,...,,,,,,,,,,
7,2026|直轄市議員|臺北市第1選舉區|黃瀞瑩,黃瀞瑩,臺北市,直轄市議員,臺北市第1選舉區,台灣民眾黨,1992-06-10,1992,女,2022官方申報，先前身分連結審核有支持；非2026戶籍認定,...,,,,,,,,,,
8,2026|直轄市議員|臺北市第1選舉區|林揚庭,林揚庭,臺北市,直轄市議員,臺北市第1選舉區,中國國民黨,,,,尚無可連結的歷史紀錄,...,,,,,,,,,,
9,2026|直轄市議員|臺北市第1選舉區|陳慈慧,陳慈慧,臺北市,直轄市議員,臺北市第1選舉區,民主進步黨,1983-11-22,1983,女,2022官方申報，先前身分連結審核有支持；非2026戶籍認定,...,,,,,,,,,,


data/processed/municipal_roles_review_round9_20261019.csv


,candidate_key,candidate_name,county_city,electoral_district,source_url,source_file,context_anchor,person_anchor,identity_review,source_type,cec_file,cec_row,cec_locality,birth_date,sex,education_level_2022,experience,decision
0,2026|縣市議員|澎湖縣第1選舉區|吳仁祥,吳仁祥,澎湖縣,澎湖縣第1選舉區,https://www.penghu.gov.tw/Rmarkun/home.jsp?id=7,data/raw/career_expansion_20261019/64937ab1e91...,馬公市民代表會第十二屆代表,代表 吳 仁 祥 60.11.07,名冊公職、縣市與單一候選人鍵核對；政府名錄以具名職務／鄉鎮及個人段落定位；歷史CEC再比對同...,government_named_person_or_role_roster_not_ful...,data/raw/cec_all_offices/103_elcand.csv,3148,馬公市第02選舉區,1971-11-07,男,大學,澎湖縣馬公市第12屆市民代表,2022_other_elected_office_supported_not_first_...
1,2026|縣市議員|澎湖縣第1選舉區|莊國輝,莊國輝,澎湖縣,澎湖縣第1選舉區,https://www.penghu.gov.tw/Rmarkun/home.jsp?id=7,data/raw/career_expansion_20261019/64937ab1e91...,馬公市民代表會第十二屆代表,副主席 莊 國 輝 79.10.10,名冊公職、縣市與單一候選人鍵核對；政府名錄以具名職務／鄉鎮及個人段落定位；歷史CEC再比對同...,government_named_person_or_role_roster_not_ful...,data/raw/cec_all_offices/103_elcand.csv,3152,馬公市第02選舉區,1990-10-10,男,大學,澎湖縣馬公市第12屆市民代表會副主席,2022_other_elected_office_supported_not_first_...
2,2026|縣市議員|澎湖縣第3選舉區|宋萬富,宋萬富,澎湖縣,澎湖縣第3選舉區,https://www.moi.gov.tw/LocalOfficial_Content.a...,data/raw/career_expansion_20261019/b587a5e46c6...,澎湖縣白沙鄉公所 宋萬富,111年度鄉鎮市長選舉,名冊公職、縣市與單一候選人鍵核對；政府名錄以具名職務／鄉鎮及個人段落定位；歷史CEC再比對同...,government_named_person_or_role_roster_not_ful...,data/raw/cec_all_offices/101_elcand.csv,481,白沙鄉,1955-08-27,男,高中(職)以下,澎湖縣白沙鄉鄉長（MOI列111年度選舉）,2022_other_elected_office_supported_not_first_...
3,2026|縣市議員|澎湖縣第4選舉區|李家萱,李家萱,澎湖縣,澎湖縣第4選舉區,https://www.moi.gov.tw/LocalOfficial_Content.a...,data/raw/career_expansion_20261019/8fb25c97b1d...,澎湖縣西嶼鄉民代表會 李家萱,111年度鄉鎮市代表選舉,名冊公職、縣市與單一候選人鍵核對；政府名錄以具名職務／鄉鎮及個人段落定位；歷史CEC再比對同...,government_named_person_or_role_roster_not_ful...,data/raw/cec_all_offices/103_elcand.csv,3202,西嶼鄉第03選舉區,1975-12-30,女,碩士,澎湖縣西嶼鄉民代表（MOI列111年度選舉）,2022_other_elected_office_supported_not_first_...
4,2026|縣市議員|新竹市第5選舉區|蔡志堅,蔡志堅,新竹市,新竹市第5選舉區,https://www.moi.gov.tw/LocalOfficial_Content.a...,data/raw/career_expansion_20261019/90edbc3b202...,蔡志堅 黨籍 無,111年度村里長選舉 學歷 大學,名冊公職、縣市與單一候選人鍵核對；政府名錄以具名職務／鄉鎮及個人段落定位；歷史CEC再比對同...,government_named_person_or_role_roster_not_ful...,data/raw/cec_all_offices/112_elcand.csv,6425,樹下里,1985-03-29,男,大學,新竹市香山區樹下里里長（MOI列111年度選舉）,2022_other_elected_office_supported_not_first_...
5,2026|縣市議員|宜蘭縣第9選舉區|曾茂照,曾茂照,宜蘭縣,宜蘭縣第9選舉區,https://www.sanshing.gov.tw/News_Content.aspx?...,data/raw/career_expansion_20261019/606002eb4a6...,代表 曾茂照,第21屆三星鄉民代表會主席,名冊公職、縣市與單一候選人鍵核對；政府名錄以具名職務／鄉鎮及個人段落定位；歷史CEC再比對同...,government_named_person_or_role_roster_not_ful...,data/raw/cec_all_offices/103_elcand.csv,291,三星鄉第03選舉區,1964-02-18,男,大學,三星鄉民代表會第18、19、20屆代表；第21屆主席；第22屆代表（頁面原文）,2022_other_elected_office_supported_not_first_...
6,2026|縣市議員|宜蘭縣第10選舉區|黃士駿,黃士駿,宜蘭縣,宜蘭縣第10選舉區,https://www.suao.gov.tw/cp.aspx?n=14770,data/raw/career_expansion_20261019/0912e18a119...,姓名： 黃士駿 職稱： 代表,長安里第21屆里長,名冊公職、縣市與單一候選人鍵核對；政府名錄以具名職務／鄉鎮及個人段落定位；歷史CEC再比對同...,government_named_person_or_role_roster_not_ful...,data/raw/cec_all_offices/103_elcand.csv,157,蘇澳鎮第03選舉區,1978-04-30,男,高中(職),蘇澳鎮民代表；曾任士敏國小及蘇澳國中家長會會長；長安里第21屆里長（依個人段落）,2022_other_elected_office_supported_not_first_...
7,2026|縣市議員|花蓮縣第3選舉區|黃宣寓,黃宣寓,花蓮縣,花蓮縣第3選舉區,https://www.ji-an.gov.tw/data/13?cate=44,data/raw/career_expansion_20261019/ae8adaeaad8...,姓名：黃宣寓 性別：男,簡歷：大學講師、花蓮電子報創辦人,名冊公職、縣市與單一候選人鍵核對；政府名錄以具名職務／鄉鎮及個人段落定位；歷史CEC再比對同...,government_named_person_or_role_roster_not_ful...,,,吉安鄉調解委員會,,男,,吉安鄉調解委員；簡歷：大學講師、花蓮電子報創辦人,sex_and_named_career_only; birth_education_fir...
8,2026|縣市議員|花蓮縣第4選舉區|吳新文,吳新文,花蓮縣,花蓮縣第4選舉區,https://www.hlyl.gov.tw/cl.aspx?n=20242,data/raw/career_expansion_20261019/e3a22c5d348...,花蓮縣玉里鎮鎮民代表會聯絡名冊,吳新文,名冊公職、縣市與單一候選人鍵核對；政府名錄以具名職務／鄉鎮及個人段落定位；歷史CEC再比對同...,government_named_person_or_role_roster_not_ful...,data/raw/cec_all_offices/103_elcand.csv,3014,玉里鎮第01選舉區,1968-07-11,男,大學,花蓮縣玉里鎮民代表（官方聯絡名冊列示）,2022_other_elected_office_supported_not_first_...
9,2026|縣市議員|澎湖縣第2選舉區|李麗美,李麗美,澎湖縣,澎湖縣第2選舉區,https://www.penghu.gov.tw/Thuhsi/ch/home.jsp?i...,data/raw/career_expansion_20261019/6b89

data/processed/candidate_evidence_round9_20261019.csv


,candidate_key,candidate_name,field,value,source_file,source_url,source_row,evidence_status,checked_date
0,2026|縣市議員|澎湖縣第1選舉區|吳仁祥,吳仁祥,profile,https://www.penghu.gov.tw/Rmarkun/home.jsp?id=7,data/raw/career_expansion_20261019/64937ab1e91...,https://www.penghu.gov.tw/Rmarkun/home.jsp?id=7,,名冊公職、縣市與單一候選人鍵核對；政府名錄以具名職務／鄉鎮及個人段落定位；歷史CEC再比對同...,2026-10-19
1,2026|縣市議員|澎湖縣第1選舉區|吳仁祥,吳仁祥,experience,澎湖縣馬公市第12屆市民代表,data/raw/career_expansion_20261019/64937ab1e91...,https://www.penghu.gov.tw/Rmarkun/home.jsp?id=7,,由具名段落審閱；不包含其他人職務，不推定2026任職法律狀態,2026-10-19
2,2026|縣市議員|澎湖縣第1選舉區|吳仁祥,吳仁祥,birth_date,1971-11-07,data/raw/cec_all_offices/103_elcand.csv,https://db.cec.gov.tw/,3148,2022官方歷史申報；名冊公職、縣市與單一候選人鍵核對；政府名錄以具名職務／鄉鎮及個人段落定...,2026-10-19
3,2026|縣市議員|澎湖縣第1選舉區|吳仁祥,吳仁祥,sex,男,data/raw/cec_all_offices/103_elcand.csv,https://db.cec.gov.tw/,3148,2022官方歷史申報；名冊公職、縣市與單一候選人鍵核對；政府名錄以具名職務／鄉鎮及個人段落定...,2026-10-19
4,2026|縣市議員|澎湖縣第1選舉區|吳仁祥,吳仁祥,education,大學,data/raw/cec_all_offices/103_elcand.csv,https://db.cec.gov.tw/,3148,2022官方歷史申報；名冊公職、縣市與單一候選人鍵核對；政府名錄以具名職務／鄉鎮及個人段落定...,2026-10-19
5,2026|縣市議員|澎湖縣第1選舉區|吳仁祥,吳仁祥,first_time_any_office,0,data/raw/cec_all_offices/103_elcand.csv,https://db.cec.gov.tw/,3148,2022官方歷史申報；名冊公職、縣市與單一候選人鍵核對；政府名錄以具名職務／鄉鎮及個人段落定...,2026-10-19
6,2026|縣市議員|澎湖縣第1選舉區|莊國輝,莊國輝,profile,https://www.penghu.gov.tw/Rmarkun/home.jsp?id=7,data/raw/career_expansion_20261019/64937ab1e91...,https://www.penghu.gov.tw/Rmarkun/home.jsp?id=7,,名冊公職、縣市與單一候選人鍵核對；政府名錄以具名職務／鄉鎮及個人段落定位；歷史CEC再比對同...,2026-10-19
7,2026|縣市議員|澎湖縣第1選舉區|莊國輝,莊國輝,experience,澎湖縣馬公市第12屆市民代表會副主席,data/raw/career_expansion_20261019/64937ab1e91...,https://www.penghu.gov.tw/Rmarkun/home.jsp?id=7,,由具名段落審閱；不包含其他人職務，不推定2026任職法律狀態,2026-10-19
8,2026|縣市議員|澎湖縣第1選舉區|莊國輝,莊國輝,birth_date,1990-10-10,data/raw/cec_all_offices/103_elcand.csv,https://db.cec.gov.tw/,3152,2022官方歷史申報；名冊公職、縣市與單一候選人鍵核對；政府名錄以具名職務／鄉鎮及個人段落定...,2026-10-19
9,2026|縣市議員|澎湖縣第1選舉區|莊國輝,莊國輝,sex,男,data/raw/cec_all_offices/103_elcand.csv,https://db.cec.gov.tw/,3152,2022官方歷史申報；名冊公職、縣市與單一候選人鍵核對；政府名錄以具名職務／鄉鎮及個人段落定...,2026-10-19


### Summary Stats

In [3]:
for p,d in D.items():
 print(p);display(pd.DataFrame({'type':d.dtypes.astype(str),'empty':d.eq('').sum()}))
r=D['data/processed/municipal_roles_review_round9_20261019.csv'];display(r[['candidate_name','birth_date','sex','education_level_2022','experience','decision']])

data/processed/candidate_master_current.csv


,type,empty
candidate_key,str,0
candidate_name,str,0
office_type,str,0
county_city,str,0
electoral_district,str,81
party_registration_raw,str,0
registration_source,str,0
registration_source_file,str,0
registration_source_page,str,0
qualification_inherited,str,0


data/processed/candidate_fields_current.csv


,type,empty
candidate_key,str,0
candidate_name,str,0
office_type,str,0
county_city,str,0
field,str,0
status,str,0
existing_value,str,25164
evidence,str,24183
note,str,23885
next_source,str,0


data/processed/council_background_20261019.csv


...

data/processed/municipal_roles_review_round9_20261019.csv


,type,empty
candidate_key,str,0
candidate_name,str,0
county_city,str,0
electoral_district,str,0
source_url,str,0
source_file,str,0
context_anchor,str,0
person_anchor,str,0
identity_review,str,0
source_type,str,0


data/processed/candidate_evidence_round9_20261019.csv


,type,empty
candidate_key,str,0
candidate_name,str,0
field,str,0
value,str,0
source_file,str,0
source_url,str,0
source_row,str,25
evidence_status,str,0
checked_date,str,0


,candidate_name,birth_date,sex,education_level_2022,experience,decision
0,吳仁祥,1971-11-07,男,大學,澎湖縣馬公市第12屆市民代表,2022_other_elected_office_supported_not_first_...
1,莊國輝,1990-10-10,男,大學,澎湖縣馬公市第12屆市民代表會副主席,2022_other_elected_office_supported_not_first_...
2,宋萬富,1955-08-27,男,高中(職)以下,澎湖縣白沙鄉鄉長（MOI列111年度選舉）,2022_other_elected_office_supported_not_first_...
3,李家萱,1975-12-30,女,碩士,澎湖縣西嶼鄉民代表（MOI列111年度選舉）,2022_other_elected_office_supported_not_first_...
4,蔡志堅,1985-03-29,男,大學,新竹市香山區樹下里里長（MOI列111年度選舉）,2022_other_elected_office_supported_not_first_...
5,曾茂照,1964-02-18,男,大學,三星鄉民代表會第18、19、20屆代表；第21屆主席；第22屆代表（頁面原文）,2022_other_elected_office_supported_not_first_...
6,黃士駿,1978-04-30,男,高中(職),蘇澳鎮民代表；曾任士敏國小及蘇澳國中家長會會長；長安里第21屆里長（依個人段落）,2022_other_elected_office_supported_not_first_...
7,黃宣寓,,男,,吉安鄉調解委員；簡歷：大學講師、花蓮電子報創辦人,sex_and_named_career_only; birth_education_fir...
8,吳新文,1968-07-11,男,大學,花蓮縣玉里鎮民代表（官方聯絡名冊列示）,2022_other_elected_office_supported_not_first_...
9,李麗美,1963-02-10,女,高中(職),澎湖縣湖西鄉民代表（官方名冊列示）,2022_other_elected_office_supported_not_first_...


### Assertions

In [4]:
c=D['data/processed/council_background_20261019.csv'];before=pd.read_csv(ROOT/'data/processed/snapshots/pre_municipal_round9_20261019/council_background_20261019.csv',dtype=str,keep_default_na=False)
assert c[['candidate_key','party_registration_raw','incumbency_2026']].equals(before[['candidate_key','party_registration_raw','incumbency_2026']])
assert c.birth_date_supported_history.ne('').sum()==703
assert (c.birth_date_supported_history.ne('')|c.birth_date_additional_official.ne('')).sum()==716
assert (c.sex_supported_history.ne('')|c.sex_additional_source_value.ne('')).sum()==717
assert (c.moi_profile_url.eq('')&c.campaign_profile_url.eq('')&c.additional_profile_url.eq('')).sum()==720
assert c.first_time_any_office_supported.eq('0').sum()==721 and c.first_time_this_office_supported.eq('0').sum()==705
assert len(r)==10 and r.birth_date.ne('').sum()==9 and r.sex.ne('').sum()==10
assert c.loc[c.candidate_name.eq('黃宣寓'),['birth_date_additional_official','education_raw','first_time_any_office_supported']].eq('').all().all()
for key in r.candidate_key:assert c.loc[c.candidate_key.eq(key),'first_time_this_office_supported'].eq('').all()
print('PASS: five exact saved outputs; counts/schema/keys/hash, named-role scope, unknowns preserved, registration and 2026 incumbency unchanged.')

PASS: five exact saved outputs; counts/schema/keys/hash, named-role scope, unknowns preserved, registration and 2026 incumbency unchanged.
